[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/carlospi2000/Notes_on_Computational_Physics/blob/main/09.F2PY_library/F2PY_library_students-5.ipynb)

# Semana 9 — Librerías de Fortran en Python con F2PY

**Física Computacional — 106018C** · Lenguaje: **Fortran y Python**

Hasta ahora cada programa se escribió en un solo lenguaje. Esta semana se combinan: los cálculos se programan en un módulo de Fortran, que se compila como una **librería**, y esa librería se importa en Python, donde se preparan los datos, se analizan los resultados y se grafican. La herramienta que hace el puente es **F2PY**, incluida en NumPy.

El notebook es corto: un ejemplo completo, las reglas que hay que conocer, una lectura opcional sobre tipos derivados y, al final, el **Taller 10**.

**Objetivos de la sesión**

1. Compilar un módulo de Fortran como librería de Python con F2PY.
2. Interpretar la firma que F2PY genera para cada procedimiento: entradas, salidas y dimensiones.
3. Comparar el tiempo de un cálculo en Python puro, en Fortran llamado desde Python y en NumPy.
4. Reconocer las restricciones de F2PY: tipos de datos, recompilación y errores que detienen el intérprete.

In [ ]:
# Preparación: bibliotecas de Python, colores del curso y herramientas de compilación
import time
import numpy as np
import matplotlib.pyplot as plt

AZUL, DORADO, ROJO, VERDE, GRIS = "#162A54", "#C9972C", "#B02A20", "#2E7D32", "#7F7F7F"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# Desde Python 3.12, F2PY compila con meson y ninja
!pip -q install meson ninja
!gfortran --version | head -n 1
print("NumPy", np.__version__)

## 1. Librerías compiladas y Python

Python es un lenguaje **interpretado**: cada instrucción de un ciclo `for` se analiza y se ejecuta una por una, con un costo fijo por iteración. Fortran es un lenguaje **compilado**: el compilador traduce el programa completo a instrucciones de máquina antes de ejecutarlo. Para ciclos largos sobre números, la diferencia de tiempo es de uno o dos órdenes de magnitud.

Una **librería compilada** es un archivo de código de máquina que otro programa puede cargar y usar. En Linux y en Colab tiene extensión `.so`; en Windows, `.pyd`. F2PY toma un archivo `.f90`, lo compila y genera además el código que traduce los datos entre los dos lenguajes. El resultado se importa con `import`, igual que `numpy`.

El esquema de trabajo es:

| Paso | Dónde | Qué se hace |
|---|---|---|
| 1 | Fortran | Se escribe un módulo con subrutinas o funciones |
| 2 | Terminal | `python -m numpy.f2py -c archivo.f90 -m nombre` compila y genera `nombre.*.so` |
| 3 | Python | `import nombre` y se llaman los procedimientos con arreglos de NumPy |

## 2. Primer ejemplo: energía cinética de un sistema de partículas

Un sistema de $N$ partículas tiene energía cinética total

$$E_k = \sum_{i=1}^{N} \frac{1}{2}\, m_i\, v_i^2 ,$$

donde $m_i$ es la masa de la partícula $i$ [kg] y $v_i$ su rapidez [m/s]. El módulo siguiente tiene dos subrutinas:

- `energia_total` recibe los arreglos de masas y rapideces y devuelve un número, $E_k$.
- `caida_libre` recibe un arreglo de tiempos y devuelve un **arreglo** con la altura de un objeto que cae desde $y_0$ con velocidad inicial nula, $y(t) = y_0 - \tfrac{1}{2} g t^2$, con $g = 9.81$ m/s².

El estilo de las declaraciones es el que F2PY entiende sin ayuda: reales `real(8)`, la dimensión `n` como argumento explícito y el `intent` de cada argumento declarado.

In [ ]:
%%writefile modulo_cinematica.f90
! ============================================================
! Módulo:    cinematica
! Propósito: ejemplo de librería de Fortran para usar desde
!            Python con F2PY.
! ============================================================
module cinematica
  implicit none
contains

  ! energia_total: energía cinética de n partículas
  ! Entradas: n, número de partículas; m(n), masas [kg]; v(n), rapideces [m/s]
  ! Salida:   ek, energía cinética total [J]
  subroutine energia_total(n, m, v, ek)
    integer, intent(in)  :: n
    real(8), intent(in)  :: m(n), v(n)
    real(8), intent(out) :: ek
    ek = 0.5d0*sum(m*v**2)                  ! operación sobre arreglos completos
  end subroutine energia_total

  ! caida_libre: altura de un objeto que parte del reposo desde y0
  ! Entradas: n, número de tiempos; t(n), tiempos [s]; y0, altura inicial [m]
  ! Salida:   y(n), alturas [m]
  subroutine caida_libre(n, t, y0, y)
    integer, intent(in)  :: n
    real(8), intent(in)  :: t(n), y0
    real(8), intent(out) :: y(n)
    real(8), parameter   :: g = 9.81d0
    y = y0 - 0.5d0*g*t**2
  end subroutine caida_libre

end module cinematica

La compilación produce mucho texto. La última línea debe indicar que se enlazó el archivo `cin.*.so`; si aparece la palabra `Error`, el problema está en el código de Fortran y el mensaje del compilador indica la línea.

In [ ]:
# Compila el módulo como la librería de Python "cin" (muestra solo las últimas líneas)
!python -m numpy.f2py -c modulo_cinematica.f90 -m cin 2>&1 | tail -n 3
!ls cin*.so

El nombre que se da con `-m` (`cin`) es el de la librería de Python; el módulo de Fortran (`cinematica`) queda dentro de ella. Por eso los procedimientos se llaman como `cin.cinematica.energia_total`. F2PY escribe la documentación de cada procedimiento, y conviene leerla antes de usarlo:

In [ ]:
import cin
print(cin.cinematica.energia_total.__doc__)

La documentación muestra cómo traduce F2PY la subrutina:

- Los argumentos `intent(in)` son los **parámetros** de la llamada: `m` y `v`, arreglos de reales de doble precisión (`'d'`).
- El argumento `intent(out)` desaparece de la lista de parámetros y se convierte en el **valor de retorno**: `ek = energia_total(m, v)`.
- La dimensión `n` pasa a ser **opcional**: F2PY la obtiene del tamaño de `m`.

Una subrutina de Fortran con argumentos `intent(out)` se usa en Python como una función que devuelve resultados.

In [ ]:
# Tres partículas: comparación con el cálculo directo en NumPy
m = np.array([1.0, 2.0, 0.5])          # masas [kg]
v = np.array([3.0, 1.0, 4.0])          # rapideces [m/s]

ek = cin.cinematica.energia_total(m, v)
print(f"E_k (Fortran) = {ek:.4f} J")
print(f"E_k (NumPy)   = {0.5*np.sum(m*v**2):.4f} J")

# Caída libre desde 20 m: la salida es un arreglo de NumPy
t = np.linspace(0.0, 2.0, 5)           # tiempos [s]
y = cin.cinematica.caida_libre(t, 20.0)
print("t [s]:", t)
print("y [m]:", y)

## 3. Cómo traduce F2PY los argumentos

| En Fortran | En Python |
|---|---|
| `real(8), intent(in) :: h` | parámetro de tipo `float` |
| `real(8), intent(in) :: y(n)` | parámetro: arreglo de NumPy (o lista, que F2PY convierte) |
| `integer, intent(in) :: n`, usado como dimensión | parámetro opcional, calculado con `len(y)` |
| `real(8), intent(out) :: s` | valor de retorno |
| `real(8), intent(out) :: y(n)` | valor de retorno: arreglo de NumPy de tamaño `n` |
| varios `intent(out)` | se devuelven en una tupla, en el orden de declaración |

F2PY convierte todos los nombres a minúsculas, porque Fortran no distingue mayúsculas de minúsculas.

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**Errores frecuentes con F2PY**

- **Recompilar sin reiniciar.** Python carga una librería una sola vez por sesión. Si se corrige el `.f90` y se recompila, `import` sigue usando la versión vieja. Después de cada recompilación, **reinicie el entorno de ejecución** (en Colab: *Entorno de ejecución → Reiniciar sesión*) y ejecute de nuevo la celda de preparación.
- **Usar `stop` dentro de la librería.** Un `stop` en Fortran termina el proceso completo, y desde Python eso significa que el intérprete se cae y se pierden todas las variables. En una librería, los casos inválidos se comunican con un argumento de salida, nunca con `stop`.
- **Declarar `real(dp)` con un parámetro del módulo.** El código que genera F2PY no conoce `dp` y la compilación falla. En los procedimientos que se llaman desde Python se usa `real(8)`.
- **Omitir el `intent`.** Sin `intent`, F2PY trata el argumento como entrada y los resultados no regresan a Python.

</div>

<div style="border-left: 5px solid #162A54; background-color: #EEF1F7; padding: 10px 14px; margin: 8px 0;">

**Arreglos de dos dimensiones.** Fortran guarda las matrices por columnas y NumPy, por defecto, por filas. F2PY hace la conversión automáticamente, pero copiando la matriz, lo que cuesta tiempo con matrices grandes. Para evitar la copia se crea el arreglo con `np.zeros((n, m), order="F")`. Este detalle será importante en el Capítulo 5, Matrices.

</div>

## 4. Comparación de tiempos

Se calcula la energía cinética de $10^6$ partículas de tres maneras: con un ciclo `for` de Python, con la librería de Fortran y con NumPy. Cada medición se repite cinco veces y se conserva la menor, porque las otras pueden estar afectadas por otros procesos del computador.

In [ ]:
def energia_python(m, v):
    """Energía cinética total con un ciclo de Python puro."""
    ek = 0.0
    for i in range(len(m)):
        ek += 0.5*m[i]*v[i]**2
    return ek

def mejor_tiempo(funcion, repeticiones=5):
    """Menor tiempo de ejecución [s] entre varias repeticiones."""
    tiempos = []
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        funcion()
        tiempos.append(time.perf_counter() - t0)
    return min(tiempos)

N = 10**6
rng = np.random.default_rng(2026)
m = rng.uniform(1.0, 2.0, N)           # masas [kg]
v = rng.normal(0.0, 300.0, N)          # componentes de velocidad [m/s]

t_py  = mejor_tiempo(lambda: energia_python(m, v))
t_f90 = mejor_tiempo(lambda: cin.cinematica.energia_total(m, v))
t_np  = mejor_tiempo(lambda: 0.5*np.sum(m*v**2))

print(f"Python puro: {t_py*1e3:9.3f} ms")
print(f"Fortran:     {t_f90*1e3:9.3f} ms   ({t_py/t_f90:5.0f} veces más rápido)")
print(f"NumPy:       {t_np*1e3:9.3f} ms   ({t_py/t_np:5.0f} veces más rápido)")

Fortran y NumPy son mucho más rápidos que el ciclo de Python por la misma razón: NumPy también es una librería compilada, escrita en C. Las operaciones con arreglos completos (`m*v**2`, `np.sum`) ejecutan el ciclo dentro del código compilado.

Entonces, ¿para qué escribir Fortran? Porque no todo cálculo se puede escribir como una operación de NumPy sobre arreglos completos. Los métodos iterativos, como Newton, Romberg o la integración de ecuaciones diferenciales, donde cada paso depende del anterior, necesitan un ciclo, y ese ciclo es rápido en Fortran y lento en Python.

## 5. Lectura opcional: tipos derivados en Fortran

*Esta sección no se evalúa.*

Un **tipo derivado** agrupa varias variables bajo un solo nombre, como un registro. Es el equivalente de una `struct` de C, y el primer paso hacia las clases. Por ejemplo, una partícula tiene masa, posición y velocidad:

```fortran
type :: particula
   real(8) :: m        ! masa [kg]
   real(8) :: x(3)     ! posición [m]
   real(8) :: v(3)     ! velocidad [m/s]
end type particula
```

Cada variable de tipo `particula` contiene las tres componentes, a las que se accede con `%`: `p%m`, `p%v(1)`. Un tipo derivado puede llevar además procedimientos propios, llamados **procedimientos ligados al tipo**, que reciben la variable como primer argumento. Así se construye en Fortran lo que en Python o C++ se llama una clase con métodos.

La celda siguiente define el tipo, le asocia un procedimiento que calcula la energía cinética y lo usa con un arreglo de partículas.

In [ ]:
%%writefile lectura_particula.f90
! ============================================================
! Programa:  lectura_particula   (lectura opcional)
! Propósito: tipo derivado con un procedimiento ligado al tipo.
! ============================================================
module modulo_particula
  implicit none
  type :: particula
     real(8) :: m                          ! masa [kg]
     real(8) :: v(3)                       ! velocidad [m/s]
   contains
     procedure :: energia                  ! procedimiento ligado al tipo
  end type particula
contains

  ! energia: energía cinética de la partícula "self"
  real(8) function energia(self)
    class(particula), intent(in) :: self
    energia = 0.5d0*self%m*sum(self%v**2)
  end function energia

end module modulo_particula

program lectura_particula
  use modulo_particula
  implicit none
  type(particula) :: gas(2)
  integer :: i

  gas(1) = particula(1.0d0, [3.0d0, 0.0d0, 0.0d0])   ! constructor del tipo
  gas(2) = particula(2.0d0, [0.0d0, 1.0d0, 1.0d0])
  do i = 1, 2
     print '(a,i0,a,f6.2,a)', 'partícula ', i, ': E_k = ', gas(i)%energia(), ' J'
  end do
end program lectura_particula

In [ ]:
!gfortran -o lectura_particula lectura_particula.f90 && ./lectura_particula

**Relación con F2PY.** F2PY no puede pasar tipos derivados entre Fortran y Python: solo maneja enteros, reales, complejos, caracteres y arreglos de esos tipos. Cuando una librería usa tipos derivados por dentro, se escribe una subrutina "envoltura" con argumentos simples, como `energia_total` de la sección 2, que recibe arreglos y arma los tipos internamente.

## 6. Taller 10: librería de integración en Fortran, usada desde Python

**Entrega:** viernes 27 de noviembre de 2026.

Este taller cierra los Capítulos 3 y 4: la integración numérica del Taller 7 se convierte en una librería de Fortran y se usa desde Python.

### A. Enunciado y requisitos

Se dispone de una función muestreada en $n$ puntos equiespaciados, $y_i = f(x_i)$ con $x_i = a + (i-1)h$, $i = 1, \dots, n$, y paso $h = (b-a)/(n-1)$. Las reglas compuestas del trapecio y de Simpson (Semana 7) aproximan $\int_a^b f(x)\,dx$ a partir del arreglo $y$:

$$T = h\left[\frac{y_1 + y_n}{2} + \sum_{i=2}^{n-1} y_i\right], \qquad
S = \frac{h}{3}\left[y_1 + y_n + 4\sum_{i\ \text{par}} y_i + 2\sum_{\substack{i\ \text{impar}\\ 3 \le i \le n-2}} y_i\right],$$

donde los índices pares $2, 4, \dots, n-1$ son los puntos medios de cada par de intervalos. Simpson requiere un número par de intervalos, es decir, $n$ impar. El error de $T$ es de orden $h^2$ y el de $S$ de orden $h^4$.

**Requisitos**

1. Las celdas del taller trabajan dentro de la carpeta `taller_f2py`, que crea la celda de la Parte B: allí se escriben el módulo y la librería, y allí los busca la verificación. En el repositorio, la carpeta `taller_f2py` contiene `modulo_integracion.f90` y este notebook. En Colab, `modulo_integracion.f90` se descarga desde el panel *Archivos*, dentro de `taller_f2py`.
2. El módulo se llama `integracion`, está en `modulo_integracion.f90` y se compila como la librería `integ`:
   `python -m numpy.f2py -c modulo_integracion.f90 -m integ`
3. El módulo contiene dos subrutinas que la celda de verificación llama directamente:

| Subrutina | Entradas | Salida |
|---|---|---|
| `trapecio` | `n`, número de puntos; `y(n)`, valores de la función; `h`, paso | `s`, integral |
| `simpson` | `n`, número de puntos, impar; `y(n)`, valores de la función; `h`, paso | `s`, integral |

   Desde Python se usan como `integ.integracion.trapecio(y, h)` y `integ.integracion.simpson(y, h)`.
4. Se usan reales `real(8)` en el módulo y `float` de 64 bits en Python. El módulo no usa `stop`.

**Qué se entrega**

- En la carpeta del repositorio: `modulo_integracion.f90` y este notebook, con la celda de verificación en 3 de 3 PASS y las respuestas de la sección D en celdas Markdown al final.
- En el Campus Virtual: el reporte de una página y el enlace a la carpeta.

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**No está permitido** usar `np.trapezoid`, `np.trapz`, `scipy.integrate` ni otras funciones de biblioteca que integren, salvo en la comparación de tiempos de la actividad 4. Las integrales se calculan con su librería de Fortran.

</div>

### B. Parte Fortran: la librería

Ejecute primero la celda siguiente, que crea la carpeta `taller_f2py` y la deja como carpeta de trabajo. Después escriba `modulo_integracion.f90` con las dos subrutinas, a partir de su trabajo del Taller 7, y compílelo.

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**Después de reiniciar el entorno** (necesario tras cada recompilación), ejecute de nuevo la celda de preparación y la celda de la carpeta de trabajo. Al reiniciar, el entorno vuelve a la carpeta inicial, fuera de `taller_f2py`.

</div>

In [ ]:
# Carpeta de trabajo del taller: aquí se escriben, compilan, importan y verifican los archivos
import os, sys
CARPETA = "taller_f2py"
if os.path.basename(os.getcwd()) != CARPETA:     # evita crear taller_f2py/taller_f2py
    os.makedirs(CARPETA, exist_ok=True)
    os.chdir(CARPETA)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())               # import integ busca la librería aquí
print("Carpeta de trabajo:", os.getcwd())

In [ ]:
%%writefile modulo_integracion.f90
! ============================================================
! Módulo:    integracion
! Propósito: reglas compuestas del trapecio y de Simpson sobre
!            valores muestreados en puntos equiespaciados.
!            Se compila como librería de Python con F2PY.
! ============================================================
module integracion
  implicit none
contains

  ! trapecio: regla compuesta del trapecio
  ! Entradas: n, número de puntos; y(n), valores de la función; h, paso
  ! Salida:   s, integral aproximada
  subroutine trapecio(n, y, h, s)
    integer, intent(in)  :: n
    real(8), intent(in)  :: y(n), h
    real(8), intent(out) :: s
    if (n < 2) then
       s = 0.0d0
    else
       s = h*(0.5d0*(y(1) + y(n)) + sum(y(2:n-1)))
    end if
  end subroutine trapecio

  ! simpson: regla compuesta de Simpson (n impar, n >= 3)
  ! Entradas: n, número de puntos (impar); y(n), valores; h, paso
  ! Salida:   s, integral aproximada. Si n es par o menor que 3
  !           devuelve NaN (no se usa stop: mataría el intérprete)
  subroutine simpson(n, y, h, s)
    integer, intent(in)  :: n
    real(8), intent(in)  :: y(n), h
    real(8), intent(out) :: s
    real(8) :: cero
    if (n < 3 .or. mod(n, 2) == 0) then
       cero = 0.0d0
       s = cero/cero                              ! NaN: entrada inválida
    else
       s = (h/3.0d0)*(y(1) + y(n) + 4.0d0*sum(y(2:n-1:2)) + 2.0d0*sum(y(3:n-2:2)))
    end if
  end subroutine simpson

end module integracion

In [ ]:
# Compila la librería "integ" (ejecute esta celda cuando su archivo esté listo)
!python -m numpy.f2py -c modulo_integracion.f90 -m integ 2>&1 | tail -n 3

### C. Parte Python: uso de la librería

Ejecute las actividades en las celdas siguientes. Cada una indica qué debe obtener.

**Actividad 1. Firma de la librería.** Importe `integ` e imprima la documentación de `trapecio`. Debe obtener una firma con `y` y `h` como parámetros, `n` como opcional y `s` como valor de retorno.

**Actividad 2. Orden de convergencia.** Calcule $\int_0^\pi \sin x\,dx = 2$ con ambas reglas para $n = 11, 21, 41, 81$. Obtenga una tabla con $n$, $h$, el error absoluto de cada regla y la razón entre errores consecutivos. Debe obtener razones cercanas a 4 para el trapecio y a 16 para Simpson.

**Actividad 3. Constante de Stefan-Boltzmann.** Calcule con Simpson la integral del Taller 8 y 9,

$$I = \int_0^\infty \frac{x^3}{e^x - 1}\,dx = \int_0^1 \frac{x(t)^3}{e^{x(t)} - 1}\,\frac{dt}{(1-t)^2}, \qquad x(t) = \frac{t}{1-t},$$

con $n = 2001$ puntos en $t \in [0, 1]$. El integrando vale cero en $t = 0$ y en $t = 1$; las muestras de los extremos se asignan con ese valor. Con $I$, obtenga $\sigma = 2\pi k_B^4 I/(h^3 c^2)$. Debe obtener $I$ con un error menor que $10^{-10}$ frente a $\pi^4/15$ y $\sigma \approx 5.670374\times10^{-8}$ W m⁻² K⁻⁴.

**Actividad 4. Tiempos.** Con $n = 10^6 + 1$ muestras de $\sin x$ en $[0, \pi]$, compare el tiempo de su trapecio de Fortran con el de una implementación del trapecio con un ciclo `for` de Python y con `np.trapezoid`. Use la función `mejor_tiempo` de la sección 4. Obtenga los tres tiempos y el factor entre Python puro y Fortran.

In [ ]:
# Actividad 1
import integ
print(integ.integracion.trapecio.__doc__)
print(integ.integracion.simpson.__doc__)

In [ ]:
# Actividad 2: orden de convergencia con sin x en [0, pi]
import numpy as np
exacto = 2.0
print(f"{'n':>4} {'h':>9} {'Error trap.':>12} {'Error Simp.':>12} {'Razón T':>8} {'Razón S':>8}")
eT_ant = eS_ant = None
for n in [11, 21, 41, 81]:
    x = np.linspace(0.0, np.pi, n)
    y = np.sin(x)
    h = x[1] - x[0]
    eT = abs(integ.integracion.trapecio(y, h) - exacto)
    eS = abs(integ.integracion.simpson(y, h) - exacto)
    rT = f"{eT_ant/eT:8.3f}" if eT_ant else f"{'--':>8}"
    rS = f"{eS_ant/eS:8.3f}" if eS_ant else f"{'--':>8}"
    print(f"{n:4d} {h:9.5f} {eT:12.3e} {eS:12.3e} {rT} {rS}")
    eT_ant, eS_ant = eT, eS

In [ ]:
# Actividad 3: constante de Stefan-Boltzmann
import numpy as np
n = 2001
t = np.linspace(0.0, 1.0, n)
dt = t[1] - t[0]
f = np.zeros_like(t)                         # extremos: el integrando vale 0
ti = t[1:-1]
x = ti/(1.0 - ti)
# para x grande, e^x desborda: el integrando es ~0 y se asigna 0
f[1:-1] = np.where(x < 700, x**3/np.expm1(np.minimum(x, 700))/(1.0 - ti)**2, 0.0)

I = integ.integracion.simpson(f, dt)
I_exacta = np.pi**4/15
print(f"I        = {I:.10f}")
print(f"pi^4/15  = {I_exacta:.10f}")
print(f"|error|  = {abs(I - I_exacta):.2e}")

kB = 1.380649e-23        # J/K
hP = 6.62607015e-34      # J s
c  = 299792458.0         # m/s
sigma = 2*np.pi*kB**4*I/(hP**3*c**2)
print(f"sigma    = {sigma:.6e} W m^-2 K^-4   (referencia: 5.670374e-08)")

In [ ]:
# Actividad 4: tiempos del trapecio
import numpy as np
N = 10**6 + 1
x = np.linspace(0.0, np.pi, N)
y = np.sin(x)
h = x[1] - x[0]

def trapecio_python(y, h):
    """Regla del trapecio con un ciclo de Python puro."""
    s = 0.0
    for i in range(1, len(y) - 1):
        s += y[i]
    return h*(0.5*(y[0] + y[-1]) + s)

t_py  = mejor_tiempo(lambda: trapecio_python(y, h))
t_f90 = mejor_tiempo(lambda: integ.integracion.trapecio(y, h))
t_np  = mejor_tiempo(lambda: np.trapezoid(y, dx=h))     # permitido solo aquí

print(f"Python puro:   {t_py*1e3:9.3f} ms")
print(f"Fortran:       {t_f90*1e3:9.3f} ms")
print(f"np.trapezoid:  {t_np*1e3:9.3f} ms")
print(f"Python / Fortran = {t_py/t_f90:.0f} veces")
print(f"Resultados: {trapecio_python(y,h):.12f}  {integ.integracion.trapecio(y,h):.12f}  {np.trapezoid(y,dx=h):.12f}")

### D. Preguntas de interpretación

Responda en celdas Markdown al final de este notebook, con los números de sus resultados:

1. **Orden de convergencia.** ¿Qué razones obtuvo en la actividad 2? ¿Qué relación tienen con el orden de cada regla?
2. **Tiempos.** ¿Cuántas veces más rápido es el trapecio de Fortran que el de Python puro? ¿Por qué `np.trapezoid` también es mucho más rápido que el ciclo de Python?
3. **Cuándo vale la pena.** Dé un ejemplo de un cálculo del curso que no se pueda escribir como una operación de NumPy sobre arreglos completos y explique por qué allí conviene una librería de Fortran.

### E. Verificación

La celda siguiente **no debe modificarse**. Importa su librería y la prueba con integrales de valor conocido. Si recompiló la librería, reinicie el entorno antes de ejecutarla. El taller exige los tres PASS.

In [ ]:
# ==================== VERIFICACIÓN DEL TALLER 10 (no modificar) ====================
import os, sys, glob
import numpy as np

resultados = []
def caso(nombre, funcion):
    try:
        detalle = funcion()
        ok = True
    except Exception as e:
        ok, detalle = False, str(e)
    resultados.append(ok)
    print(f"[{'PASS' if ok else 'FAIL'}] {nombre}" + (f" -> {detalle}" if detalle else ""))

# La verificación siempre trabaja dentro de la carpeta del taller
CARPETA = "taller_f2py"
if os.path.basename(os.getcwd()) != CARPETA:
    if not os.path.isdir(CARPETA):
        raise SystemExit(f"No existe la carpeta {CARPETA}: ejecute la celda de la Parte B.")
    os.chdir(CARPETA)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Verificando en:", os.getcwd(), "\n")

def libreria():
    if not os.path.isfile("modulo_integracion.f90"):
        raise ValueError(f"no existe modulo_integracion.f90 en {CARPETA}")
    so = glob.glob("integ.*.so") + glob.glob("integ.*.pyd")
    if not so:
        raise ValueError(f"no se encontró la librería integ en {CARPETA}; compile con F2PY")
    so = max(so, key=os.path.getmtime)                # la más reciente, si hay varias
    if os.path.getmtime(so) < os.path.getmtime("modulo_integracion.f90"):
        raise ValueError("la librería es anterior a modulo_integracion.f90; recompile y reinicie")
    import integ
    if os.path.abspath(integ.__file__) != os.path.abspath(so):
        raise ValueError(f"la librería cargada es {integ.__file__}, no la de {CARPETA}; reinicie el entorno")
    if not hasattr(integ, "integracion"):
        raise ValueError("la librería no contiene el módulo integracion")
    return integ.integracion

def muestras(f, a, b, n):
    x = np.linspace(a, b, n)
    return f(x), x[1] - x[0]

def c1():
    lib = libreria()
    y, h = muestras(lambda x: 3*x + 1, 0.0, 2.0, 7)        # recta: el trapecio es exacto
    if abs(lib.trapecio(y, h) - 8.0) > 1e-12:
        raise ValueError("trapecio no es exacto para una recta")
    y, h = muestras(np.sin, 0.0, np.pi, 21)
    ref = h*(0.5*(y[0] + y[-1]) + y[1:-1].sum())
    if abs(lib.trapecio(y, h) - ref) > 1e-13:
        raise ValueError("trapecio no coincide con la regla compuesta")
    return "exacto para una recta y correcto con sin x"

def c2():
    lib = libreria()
    y, h = muestras(lambda x: x**3 - 2*x, 0.0, 2.0, 5)     # cúbica: Simpson es exacto
    if abs(lib.simpson(y, h) - 0.0) > 1e-12:
        raise ValueError("simpson no es exacto para una cúbica")
    y, h = muestras(np.sin, 0.0, np.pi, 21)
    ref = h/3*(y[0] + y[-1] + 4*y[1:-1:2].sum() + 2*y[2:-1:2].sum())
    if abs(lib.simpson(y, h) - ref) > 1e-13:
        raise ValueError("simpson no coincide con la regla compuesta")
    return "exacto para una cúbica y correcto con sin x"

def c3():
    lib = libreria()
    eT, eS = [], []
    for n in [11, 21, 41, 81]:
        y, h = muestras(np.sin, 0.0, np.pi, n)
        eT.append(abs(lib.trapecio(y, h) - 2)); eS.append(abs(lib.simpson(y, h) - 2))
    rT = np.array(eT[:-1])/np.array(eT[1:]); rS = np.array(eS[:-1])/np.array(eS[1:])
    if np.any(abs(rT - 4) > 0.1) or np.any(abs(rS - 16) > 0.5):
        raise ValueError("las razones de convergencia no corresponden a órdenes 2 y 4")
    t = np.linspace(0.0, 1.0, 2001)
    f = np.zeros_like(t)
    x = t[1:-1]/(1 - t[1:-1])
    f[1:-1] = np.where(x < 700, x**3/np.expm1(np.minimum(x, 700))/(1 - t[1:-1])**2, 0.0)
    I = lib.simpson(f, t[1] - t[0])
    if abs(I - np.pi**4/15) > 1e-10:
        raise ValueError("la integral de Planck no alcanza la precisión pedida")
    return f"razones T ≈ {rT.mean():.2f}, S ≈ {rS.mean():.1f}; I = {I:.10f}"

caso("1. Regla del trapecio", c1)
caso("2. Regla de Simpson", c2)
caso("3. Orden de convergencia e integral de Planck", c3)
print(f"\nResultado: {sum(resultados)} de {len(resultados)} PASS")

## Respuestas de la sección D

**1. Orden de convergencia.** Al duplicar el número de intervalos (h → h/2) las razones entre errores consecutivos fueron ≈ 4.005, 4.001 y 4.000 para el trapecio, y ≈ 16.14, 16.04 y 16.01 para Simpson (errores en $n=81$: $2.57\times10^{-4}$ y $2.64\times10^{-8}$). Si el error es de orden $h^p$, al reducir h a la mitad se divide por $2^p$: $2^2 = 4$ confirma que el trapecio es de orden $h^2$ y $2^4 = 16$ confirma que Simpson es de orden $h^4$.

**2. Tiempos.** El trapecio de Fortran fue ____ veces más rápido que el de Python puro (Python: ____ ms; Fortran: ____ ms; `np.trapezoid`: ____ ms). *(Complete con los valores de su actividad 4; en el computador de las diapositivas, para $E_k$ con $10^6$ datos, el factor fue ≈ 240.)* El ciclo de Python es interpretado: cada iteración se analiza y se ejecuta una por una, con un costo fijo por iteración. `np.trapezoid` también es rápido porque NumPy es una librería compilada, escrita en C: la suma sobre el arreglo completo se ejecuta dentro del código compilado, sin pasar por el intérprete en cada elemento.

**3. Cuándo vale la pena.** Un ejemplo es la tabla de Romberg con su criterio de parada (o Newton-Raphson, o RK4 / Euler para una EDO): cada paso depende del resultado del anterior (cada iteración usa el valor anterior y se detiene según una condición), por lo que no se puede escribir como una operación sobre un arreglo completo y exige un ciclo. Ese ciclo es lento en Python (interpretado) y rápido en Fortran (compilado), así que conviene escribirlo en una librería de Fortran y llamarlo desde Python.